<a href="https://colab.research.google.com/github/OpenForest4D/ALS_Notebooks/blob/main/USGS_3DEP_lidar_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# USGS 3DEP Lidar Point Cloud Data Download (Entwine Point Tiles)

This notebook enables users to discover, subset and download [USGS 3D Elevation Program](https://www.usgs.gov/3d-elevation-program) (3DEP) lidar point cloud data, with additional options to visualize the selection. The dataset is publicly available and provided in Entwine Point Tiles (EPT) format, a lossless, full-density, streamable octree based on LASzip (LAZ) encoding. Data can be accessed via the USGS AWS bucket at https://registry.opendata.aws/usgs-lidar/.

Notebook workflow


1. Install and import dependencies
2. Configuration
3. Select AOI and dataset on Interactive USGS 3DEP map
4. Subset, filter and download the selected dataset
5. Quick preview/visualize dataset (optional)
6. Interactive 3D visualization (optional)


> ⚠️ **This notebook is built for Google Colab** (it installs PDAL via `apt-get`, which requires Colab's root-privileged Linux VM). It will not run as-is in a local Jupyter install - open it at [colab.research.google.com](https://colab.research.google.com).

Authors: Viswanath Nandigam, Minh Phan
info@openforest4d.org

Work is part of the OpenForest4D project supported by NSF award numbers [2409885](https://www.nsf.gov/awardsearch/show-award/?AWD_ID=2409885), 2409886 & 2409887

### Step 1 - Install and import dependencies

In [ ]:
# Install dependencies
!apt-get install -y pdal > /dev/null
!pip install -q ipyleaflet geopandas shapely pyproj laspy lazrs

# Core geospatial / mapping
from ipyleaflet import Map, DrawControl, GeoJSON, LayersControl, ScaleControl, basemaps
import geopandas as gpd                  # Reads and spatially queries the EPT boundary index
import pyproj                            # Geodesic area calculation + CRS transforms
from pyproj import Transformer
from shapely.geometry import shape

# Point cloud processing
import json
import os
import shutil
import subprocess

# UI
import ipywidgets as widgets
from IPython.display import display

### Step 2 - Configuration

In [3]:
# USGS 3DEP metadata and spatial boundaries file location
RESOURCES_URL = "https://usgs.entwine.io/boundaries/resources.geojson"

#Restrict user spatial AOI selection size for resource management
MAX_AREA_SQKM = 3                          # Set cap at 3 sq kms
MAX_AREA_SQM = MAX_AREA_SQKM * 1_000_000
POINT_WARN_THRESHOLD = 20_000_000            # Warn user if number of points exceed 20 Million

# Globals populated by the map / dropdown interaction in the next cell
USER_BBOX = None       # (minx, miny, maxx, maxy) in EPSG:4326
USER_GEOM = None       # shapely geometry of the drawn AOI
USER_EPT_URL = None    # ept.json URL of the selected dataset
USER_DATASET = None    # dataset name of the selection

if shutil.which("pdal") is None:
    raise EnvironmentError(
        "The 'pdal' CLI was not found on PATH. Run the install cell above first "
        "(it apt-installs PDAL), and make sure you're running this notebook in "
        "Google Colab."
    )
print("PDAL CLI found at:", shutil.which("pdal"))

PDAL CLI found at: /usr/bin/pdal


### Step 3 - Select a spatial area of interest (AOI) and USGS 3DEP Dataset

**Instructions**

1. Wait for the map and the dataset coverage boundaries to load.
2. Zoom in to an area with coverage (the blue outlines are USGS 3DEP EPT datasets).
3. Use the rectangle tool to draw a box — **drawing a new box replaces the old one**.
4. If the box is larger than **3 km²** (defined by MAX_AREA_SQKM above) it will be rejected automatically; re-draw a smaller AOI.
5. Choose a dataset from the dropdown that appears below the map.

In [ ]:
# 1. Load the USGS 3DEP EPT dataset boundaries -------------------------------
# GDAL's default fetcher (used internally by gpd.read_file on a URL) sends no
# User-Agent and may get a 403 error, so fetch the bytes ourselves first.

import requests
import io

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; USGS-3DEP-notebook/1.0)",
    "Referer": "https://usgs.entwine.io/",
}

# hobuinc/usgs-lidar mirrors the same resources.geojson used by usgs.entwine.io
FALLBACK_URL = "https://raw.githubusercontent.com/hobuinc/usgs-lidar/master/boundaries/resources.geojson"

gdf = None
for url in (RESOURCES_URL, FALLBACK_URL):
    try:
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
        gdf = gpd.read_file(io.BytesIO(resp.content))
        if url != RESOURCES_URL:
            print(f"Note: primary URL failed, loaded boundaries from mirror: {url}")
        break
    except Exception as e:
        last_error = e

if gdf is None:
    raise RuntimeError(
        f"Failed to load the USGS 3DEP boundary index from {RESOURCES_URL} "
        f"or its mirror.\nLast error: {last_error}"
    )

for required_col in ("name", "url"):
    if required_col not in gdf.columns:
        raise RuntimeError(f"Unexpected schema in resources.geojson - missing '{required_col}' column")

print(f"Loaded {len(gdf):,} USGS 3DEP EPT dataset boundaries.")

# A simplified copy purely for fast rendering on the map. All intersection /
# area calculations below always use the original, unsimplified `gdf`.
gdf_display = gdf.copy()
gdf_display["geometry"] = gdf_display.geometry.simplify(0.002, preserve_topology=True)
boundaries_geojson = json.loads(gdf_display.to_json())

# 2. Build the interactive map (ipyleaflet)
m = Map(center=(40, -100), zoom=4, basemap=basemaps.USGS.USTopo, scroll_wheel_zoom=True)
m.layout.height = "560px"
m.add_control(LayersControl(position="topright"))
m.add_control(ScaleControl(position="bottomleft"))

boundaries_layer = GeoJSON(
    data=boundaries_geojson,
    style={"color": "#be2b63", "weight": 1, "fillOpacity": 0.08},
    hover_style={"fillOpacity": 0.3},
    name="USGS 3DEP Data Coverage",
)
m.add_layer(boundaries_layer)

# Rectangle-only draw toolbar - that's how the AOI is picked
draw_control = DrawControl(
    rectangle={"shapeOptions": {"color": "#f7b731", "weight": 3, "fillOpacity": 0.1}},
    polyline={}, polygon={}, circle={}, marker={}, circlemarker={},
)
m.add_control(draw_control)
aoi_layer = None  # holds the current AOI highlight layer, replaced on each draw

# 3. UI elements
info_out = widgets.Output()
dataset_dropdown = widgets.Dropdown(
    options={"\u2014 draw a box on the map first \u2014": None},
    description="Dataset:",
    layout={"width": "max-content"},
)

def _geod_area_sqm(geom):
    geod = pyproj.Geod(ellps="WGS84")
    return abs(geod.geometry_area_perimeter(geom)[0])

def handle_draw(target, action, geo_json):
    global USER_BBOX, USER_GEOM, USER_EPT_URL, USER_DATASET, aoi_layer

    with info_out:
        info_out.clear_output()

        if action != "created":
            return

        geom = shape(geo_json["geometry"])
        area_sqm = _geod_area_sqm(geom)
        area_sqkm = area_sqm / 1_000_000

        # Enforce the AOI size cap
        if area_sqm > MAX_AREA_SQM:
            draw_control.clear()  # remove the oversized rectangle from the map
            USER_BBOX = USER_GEOM = USER_EPT_URL = USER_DATASET = None
            dataset_dropdown.options = {"\u2014 selection too large, redraw \u2014": None}
            print(
                f"\u274c Selection is {area_sqkm:.3f} km\u00b2, which exceeds the "
                f"{MAX_AREA_SQKM} km\u00b2 limit. Draw a smaller box."
            )
            return

        USER_BBOX = geom.bounds
        USER_GEOM = geom
        USER_EPT_URL = None
        USER_DATASET = None

        # Highlight the accepted AOI on the map
        if aoi_layer is not None:
            try:
                m.remove_layer(aoi_layer)
            except Exception:
                pass
        aoi_layer = GeoJSON(
            data=geo_json,
            style={"color": "#f7b731", "weight": 3, "fillOpacity": 0.1},
            name="Selected AOI",
        )
        m.add_layer(aoi_layer)

        # Find intersecting datasets
        intersecting = gdf[gdf.intersects(geom)]

        if intersecting.empty:
            print(f"AOI: {area_sqkm:.3f} km\u00b2 - no USGS 3DEP datasets intersect this area.")
            dataset_dropdown.options = {"\u2014 no coverage here \u2014": None}
            return

        options = {}
        for _, row in intersecting.iterrows():
            label = row["name"]
            # Rough point-count estimate for this AOI, informational only.
            try:
                dataset_area_sqm = _geod_area_sqm(row.geometry)
                overlap_area_sqm = _geod_area_sqm(row.geometry.intersection(geom))
                total_count = row.get("count", None)
                if total_count and dataset_area_sqm > 0:
                    # we increase estimate by 1.5 to be on conservative side
                    # given uneven distribution of lidar points in collection
                    est_points = int(total_count * (overlap_area_sqm / dataset_area_sqm) * 1.5)
                    flag = " \u26a0\ufe0f large" if est_points > POINT_WARN_THRESHOLD else ""
                    label = f"{row['name']}  (~{est_points:,} pts{flag})"
            except Exception:
                pass
            options[label] = row["url"]

        dataset_dropdown.options = options
        print(f"AOI: {area_sqkm:.3f} km\u00b2 - {len(intersecting)} dataset(s) found. Pick one below.")


def handle_dropdown(change):
    global USER_EPT_URL, USER_DATASET
    if change["name"] == "value" and change["new"]:
        USER_EPT_URL = change["new"]
        USER_DATASET = next(k for k, v in dataset_dropdown.options.items() if v == change["new"])
        with info_out:
            print(f"Selected dataset: {USER_DATASET}")


draw_control.on_draw(handle_draw)
dataset_dropdown.observe(handle_dropdown, names="value")

display(m)
display(dataset_dropdown)
display(info_out)

### Step 4 - Subset, filter and download the selected dataset

Uses the USER_BBOX and USER_EPT_URL set in the previous cell to build a PDAL pipeline: readers.ept clips the point cloud to your AOI, a filters.expression drops common noise/overlap classes, and writers.las (with LASzip compression) writes a compressed **`.laz`** file.

**Filtering note:** removes Classification 7 (Low Noise), 18 (High Noise), and 12 (Overlap). Classification 1 (Unclassified) is kept.

In [ ]:
if not USER_BBOX or not USER_EPT_URL:
    raise ValueError("Draw an AOI and select a dataset in the previous cell before running this one.")

minx, miny, maxx, maxy = USER_BBOX

# PDAL's EPT reader expects bounds in the EPT resource's native CRS, which for
# USGS 3DEP data is EPSG:3857 (Web Mercator).
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
minx_3857, miny_3857 = transformer.transform(minx, miny)
maxx_3857, maxy_3857 = transformer.transform(maxx, maxy)
bounds_str = f"([{minx_3857}, {maxx_3857}], [{miny_3857}, {maxy_3857}])"

safe_name = USER_DATASET.split("  (")[0].replace(" ", "_")
output_file = f"{safe_name}_subset.laz"

pipeline = {
    "pipeline": [
        {
            "type": "readers.ept",
            "filename": USER_EPT_URL,
            "bounds": bounds_str,
        },
        {
            "type": "filters.expression",
            "expression": "Classification != 7 && Classification != 18 && Classification != 12",
        },
        {
            "type": "writers.las",
            "filename": output_file,
            "compression": "laszip",  # forces compressed LAZ output
        },
    ]
}

with open("pipeline.json", "w") as f:
    json.dump(pipeline, f, indent=2)

print(f"Downloading from {USER_EPT_URL} ...")
result = subprocess.run(["pdal", "pipeline", "pipeline.json"], capture_output=True, text=True)

if os.path.exists("pipeline.json"):
    os.remove("pipeline.json")

if result.returncode == 0:
    size_mb = os.path.getsize(output_file) / (1024 * 1024)
    output_path = os.path.abspath(output_file)
    print(f"\u2705 Saved to Colab runtime disk: {output_path} ({size_mb:.2f} MB)")
    print("Use the button below to save a copy to your own computer.")
else:
    print(result.stderr)
    raise RuntimeError("PDAL pipeline failed. See stderr above.")

# Download the .laz file from the Colab VM to your local machine.
# Triggers a normal browser file-save dialog/download.
from google.colab import files

download_button = widgets.Button(
    description=f"Download {output_file}",
    icon="download",
    button_style="success",
    layout={"width": "max-content"},
)
download_out = widgets.Output()


def _on_download_click(_):
    with download_out:
        download_out.clear_output()
        if not os.path.exists(output_file):
            print(f"\u274c {output_file} not found - re-run the download cell above first.")
            return
        files.download(output_file)


download_button.on_click(_on_download_click)
display(download_button, download_out)


### Step 5 - Quick Preview / visualize dataset (optional)

Reads the downloaded .laz back with laspy and shows a fast top-down elevation scatter plot.

In [ ]:
import laspy
import numpy as np

las = laspy.read(output_file)
print(f"Points: {len(las.points):,}")
print(f"X range: {las.x.min():.1f} to {las.x.max():.1f}")
print(f"Y range: {las.y.min():.1f} to {las.y.max():.1f}")
print(f"Z range: {las.z.min():.1f} to {las.z.max():.1f}")

import matplotlib.pyplot as plt

step = max(1, len(las.points) // 200_000)  # decimate for a responsive plot
fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(las.x[::step], las.y[::step], c=las.z[::step], cmap="viridis", s=1)
ax.set_aspect("equal")
#remove point estimate + warning flag in dataset name if present and add label
ax.set_title(USER_DATASET.split("  (")[0] if USER_DATASET else "Downloaded AOI")
plt.colorbar(sc, label="Elevation (m)")
plt.show()

### Step 6 - Interactive 3D visualization (optional)

In [ ]:
import plotly.graph_objects as go
print("Preparing Interactive 3D Scene...")

# Read the las file
las_3d = laspy.read(output_file)
total_points = len(las_3d.x)

# Smart Decimation: Target ~250,000 points for optimal performance
target_points = 250_000

# Smart detect step value
step = 1 if total_points <= target_points else int(total_points / target_points)

# Apply the dynamic step AND normalize the coordinates to start near (0,0,0)
dx = las_3d.x[::step] - np.min(las_3d.x)
dy = las_3d.y[::step] - np.min(las_3d.y)
dz = las_3d.z[::step] # Z is usually already a small number (elevation), so we leave it alone

print(f"Original points: {total_points:,}")
print(f"Smart Step applied: {step} (Rendering {len(dx):,} points)")

# Create the Plotly 3D Scatter Plot
fig = go.Figure(data=[go.Scatter3d(
    x=dx,
    y=dy,
    z=dz,
    mode='markers',
    marker=dict(
        size=1.5,
        color=dz,
        colorscale='Earth',
        colorbar=dict(title="Elevation", titleside="right"),
        opacity=1.0
    )
)])

# Finalize layout
fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(aspectmode='data') # Prevents Z-axis stretching
)

print("Done! Left-click to rotate, right-click to pan, and scroll to zoom.")
fig.show()

### Data Citation

USGS 3DEP LiDAR Point Clouds was accessed on `DATE` from https://registry.opendata.aws/usgs-lidar.

### Funding acknowledgement

NSF award numbers [2409885](https://www.nsf.gov/awardsearch/show-award/?AWD_ID=2409885), 2409886 & 2409887
